# Youtube Chatbot Extension

In [ ]:
pip install openai


In [ ]:
pip install google-genai


In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"



In [ ]:
!pip install -q youtube-transcript-api langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Replace OpenAI with Gemini
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

#step 1a - Indexing(Document Ingestion)

In [ ]:
import textwrap


In [ ]:
 video_id = "d2kxUVwWWwU"  # Only the YouTube video ID

try:
    # Get transcript (English)
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    )

    # Convert transcript to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(textwrap.fill(transcript, width=100))

except TranscriptsDisabled:
    print("No captions available for this video.")

#Step 1b - Indexing (Text Spliting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=250
)

chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[1]

#Step 1c & 1d - Indexing(Embedding Generation and storing in VectorStore)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['8ad84c64-44a1-4fab-9637-7b00d6a58df2'])

#Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retriever.invoke("What is the purpose of this video?")

#Step 3- Augmentation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}

Question: {question}
""",
    input_variables=["context", "question"]
)

In [ ]:
question = "What is the purpose of this video?"

retrieved_docs = retriever.invoke(question)

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})



#Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)


In [ ]:
answer.content

#Building Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

from langchain_core.output_parsers import StrOutputParser


def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text


parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
)

parallel_chain.invoke("overall what topics this video covered?")

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("overall what topics this video covered?")